# TEKNOFEST Sağlıkta YZ — Overfitting'e karşı NaN modeli (Colab)

Strateji: 350 gösterge kolonu yerine **birkaç özet eksiklik özelliği**, **düzenlileştirilmiş LightGBM**, **early stopping** ve **stratified CV**. Notebook ayrıca göstergeli/göstergesiz iki modeli kıyaslayıp **train–val AUC farkını** (overfitting göstergesi) raporlar.

**Kullanım:** Sol menüden hücreleri sırayla çalıştır. 2. hücrede 4 CSV'yi yükle.

In [ ]:
# 1) Kurulum + importlar
!pip -q install lightgbm --upgrade
import pandas as pd, numpy as np, re
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
print('lightgbm', lgb.__version__)

In [ ]:
# 2) Veriyi yükle
# Yöntem A: dosyaları elle yükle (YARISMA_TRAIN_*.csv dosyalarini sec)
from google.colab import files
uploaded = files.upload()        # MASTER (ve istersen CFTR/KANSER/PAH) dosyalarini sec
DATA_DIR = '/content'

# Yöntem B (alternatif): Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/teknofest'

TRAIN_FILE = f'{DATA_DIR}/YARISMA_TRAIN_MASTER.csv'   # az veri oldugu icin MASTER onerilir
df = pd.read_csv(TRAIN_FILE)
print('shape:', df.shape)

In [ ]:
# 3) Özellik mühendisliği (overfitting'i azaltan kisim)
ID_COL, LABEL_COL = 'Variant_ID', 'Label'
y = df[LABEL_COL].astype(int)
X = df.drop(columns=[c for c in (ID_COL, LABEL_COL) if c in df.columns])

# 3a) cok bos kolonlari at (>%90 NaN = cogunlukla gurultu)
nan_rate = X.isna().mean()
drop_cols = nan_rate[nan_rate > 0.90].index.tolist()
X = X.drop(columns=drop_cols)
print('atilan kolon (>%90 NaN):', len(drop_cols))

# 3b) kategorikleri category dtype yap -> LightGBM ham isler, NaN doldurma yok
cat_cols = X.select_dtypes('object').columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')

# 3c) OZET eksiklik ozellikleri: 350 gosterge yerine sadece birkac tane
def prefix(c):
    m = re.match(r'[A-Za-z]+', c); return m.group() if m else 'X'
groups = {}
for c in X.columns:
    groups.setdefault(prefix(c), []).append(c)

def make_miss_feats(frame):
    mf = pd.DataFrame(index=frame.index)
    mf['nan_total_frac'] = frame.isna().mean(axis=1)
    for g, cols in groups.items():
        cc = [c for c in cols if c in frame.columns]
        mf[f'nan_{g}_count'] = frame[cc].isna().sum(axis=1)
    return mf

miss_feats = make_miss_feats(X)
print('ozet eksiklik ozelligi sayisi:', miss_feats.shape[1], '->', list(miss_feats.columns))

In [ ]:
# 4) Düzenlileştirilmiş model + CV fonksiyonu (train-val farkini raporlar)
PARAMS = dict(
    objective='binary', metric='auc',
    n_estimators=3000, learning_rate=0.02,
    num_leaves=31, max_depth=5,          # agaci siglig-tut
    min_child_samples=60,                # yaprak basina min ornek -> ezberi onler
    reg_alpha=1.0, reg_lambda=2.0,       # L1/L2 duzenlilestirme
    feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1,
    class_weight='balanced',             # 2149 vs 782 dengesizligi
    verbosity=-1, n_jobs=-1,
)

def run_cv(X_in, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr_aucs, va_aucs, best_iters = [], [], []
    for tr, va in skf.split(X_in, y):
        Xtr, Xva = X_in.iloc[tr], X_in.iloc[va]
        ytr, yva = y.iloc[tr], y.iloc[va]
        m = lgb.LGBMClassifier(**PARAMS)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)],
              callbacks=[lgb.early_stopping(150, verbose=False)])
        tr_aucs.append(roc_auc_score(ytr, m.predict_proba(Xtr)[:, 1]))
        va_aucs.append(roc_auc_score(yva, m.predict_proba(Xva)[:, 1]))
        best_iters.append(m.best_iteration_ or PARAMS['n_estimators'])
    tr, va = np.mean(tr_aucs), np.mean(va_aucs)
    print(f'  train AUC={tr:.4f}  val AUC={va:.4f}  fark(overfit)={tr-va:.4f}  ~iter={int(np.mean(best_iters))}')
    return va, tr - va, int(np.mean(best_iters))

In [ ]:
# 5) İki konfigürasyonu kıyasla
print('A) sadece ham ozellik (native NaN, ozet eksiklik YOK):')
vaA, gapA, itA = run_cv(X, y)

print('B) ham + ozet eksiklik ozellikleri:')
XB = pd.concat([X, miss_feats], axis=1)
vaB, gapB, itB = run_cv(XB, y)

use_B = (vaB > vaA) and (gapB <= gapA + 0.02)   # daha iyi val + asiri overfit etmiyorsa
print('\nSECIM:', 'B (ozet eksiklik faydali)' if use_B else 'A (gostergesiz daha guvenli)')
final_X   = XB if use_B else X
best_iter = itB if use_B else itA

In [ ]:
# 6) Final modeli eğit + en önemli özellikler
final_params = {**PARAMS, 'n_estimators': max(200, best_iter)}
model = lgb.LGBMClassifier(**final_params)
model.fit(final_X, y)

imp = pd.Series(model.feature_importances_, index=final_X.columns).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).iloc[::-1].plot.barh(figsize=(7, 6))
plt.title('En onemli 20 ozellik'); plt.tight_layout(); plt.show()

In [ ]:
# 7) Test setinde tahmin (test CSV'si gelince yorumu kaldir)
# TEST_FILE = f'{DATA_DIR}/YARISMA_TEST_MASTER.csv'
# test = pd.read_csv(TEST_FILE)
# ids = test[ID_COL] if ID_COL in test.columns else pd.RangeIndex(len(test))
# Xt = test.drop(columns=[c for c in (ID_COL, LABEL_COL) if c in test.columns])
# Xt = Xt.drop(columns=[c for c in drop_cols if c in Xt.columns], errors='ignore')
# for c in cat_cols:
#     if c in Xt.columns: Xt[c] = Xt[c].astype('category')
# if use_B:
#     Xt = pd.concat([Xt, make_miss_feats(Xt)], axis=1)
# Xt = Xt.reindex(columns=final_X.columns, fill_value=np.nan)
# proba = model.predict_proba(Xt)[:, 1]
# sub = pd.DataFrame({ID_COL: ids, 'Label': (proba >= 0.5).astype(int)})
# sub.to_csv('submission.csv', index=False)
# print(sub['Label'].value_counts()); print('submission.csv kaydedildi')
# files.download('submission.csv')